# Telecom Churn Prediction Model
**Random Forest Classifier — Full Pipeline**

Pipeline: Data Load → EDA → Preprocessing → Train/Test Split → Model Training → Evaluation → Threshold Tuning → Predictions

**Dataset:** `telecom_customers.csv` — 6,418 customers  
- Training data: `Stayed` + `Churned` rows (5,907 customers)  
- Prediction data: `Joined` rows (411 new customers)


---
## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    ConfusionMatrixDisplay
)
from sklearn.preprocessing import LabelEncoder
import joblib

print('All libraries imported successfully.')

---
## 2. Load Data

In [ ]:
# Single CSV contains all customers — Stayed, Churned, and Joined
df = pd.read_csv('telecom_customers.csv')

print(f'Dataset shape : {df.shape}')
print(f'\nCustomer Status breakdown:')
print(df['Customer_Status'].value_counts())
df.head()

---
## 3. Exploratory Data Analysis

In [ ]:
# Missing values check
missing = df.isnull().sum()
missing = missing[missing > 0]
if len(missing) > 0:
    print('Columns with missing values:')
    print(missing)
else:
    print('No missing values found.')

In [ ]:
# Churn rate among Stayed + Churned customers
train_df = df[df['Customer_Status'].isin(['Stayed', 'Churned'])]
churn_rate = (train_df['Customer_Status'] == 'Churned').mean() * 100
print(f'Churn Rate : {churn_rate:.2f}%')
print(f'Stayed     : {(train_df["Customer_Status"] == "Stayed").sum()}')
print(f'Churned    : {(train_df["Customer_Status"] == "Churned").sum()}')

In [ ]:
# Customer status distribution plot
fig, ax = plt.subplots(figsize=(6, 4))
counts = df['Customer_Status'].value_counts()
colors = ['#2ecc71', '#e74c3c', '#3498db']
ax.bar(counts.index, counts.values, color=colors, edgecolor='white')
ax.set_title('Customer Status Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Customer Status')
ax.set_ylabel('Count')
for i, v in enumerate(counts.values):
    ax.text(i, v + 30, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('churn_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Churn rate by Contract type
contract_churn = train_df.groupby('Contract')['Customer_Status'].apply(
    lambda x: (x == 'Churned').mean() * 100
).reset_index()
contract_churn.columns = ['Contract', 'Churn_Rate']

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(contract_churn['Contract'], contract_churn['Churn_Rate'], color='#e74c3c', edgecolor='white')
ax.set_title('Churn Rate by Contract Type', fontsize=13, fontweight='bold')
ax.set_xlabel('Contract Type')
ax.set_ylabel('Churn Rate (%)')
for i, v in enumerate(contract_churn['Churn_Rate']):
    ax.text(i, v + 0.5, f'{v:.1f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('churn_by_contract.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Data Preprocessing

In [ ]:
# Split into training data (Stayed + Churned) and prediction data (Joined)
train_data = df[df['Customer_Status'].isin(['Stayed', 'Churned'])].copy()
join_data  = df[df['Customer_Status'] == 'Joined'].copy()

print(f'Training data  : {train_data.shape[0]} rows')
print(f'Prediction data: {join_data.shape[0]} rows')

In [ ]:
# Drop columns not used for prediction
drop_cols = ['Customer_ID', 'Churn_Category', 'Churn_Reason']
train_data = train_data.drop(columns=drop_cols)

# Columns to label encode
columns_to_encode = [
    'Gender', 'Married', 'State', 'Value_Deal', 'Phone_Service', 'Multiple_Lines',
    'Internet_Service', 'Internet_Type', 'Online_Security', 'Online_Backup',
    'Device_Protection_Plan', 'Premium_Support', 'Streaming_TV', 'Streaming_Movies',
    'Streaming_Music', 'Unlimited_Data', 'Contract', 'Paperless_Billing',
    'Payment_Method'
]

# Fit label encoders on training data
label_encoders = {}
for col in columns_to_encode:
    le = LabelEncoder()
    train_data[col] = le.fit_transform(train_data[col])
    label_encoders[col] = le

# Encode target variable
train_data['Customer_Status'] = train_data['Customer_Status'].map({'Stayed': 0, 'Churned': 1})

print('Preprocessing complete.')
print(f'Features : {train_data.shape[1] - 1}')
train_data.head()

---
## 5. Train / Test Split

In [ ]:
X = train_data.drop('Customer_Status', axis=1)
y = train_data['Customer_Status']

# stratify=y preserves the 27% churn ratio in both train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set : {X_train.shape[0]} rows')
print(f'Test set     : {X_test.shape[0]} rows')
print(f'Churn ratio  : {y.mean() * 100:.2f}%')

---
## 6. Train Random Forest Model

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced'  # handles class imbalance (27% churn vs 73% stayed)
)

rf_model.fit(X_train, y_train)
print('Model training complete.')

---
## 7. Model Evaluation

In [ ]:
# Predictions
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]  # probability scores

# Metrics
auc_score = roc_auc_score(y_test, y_prob)

print('=' * 50)
print('       MODEL PERFORMANCE SUMMARY')
print('=' * 50)
print(f'  ROC-AUC Score : {auc_score:.4f}')
print('=' * 50)
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Stayed', 'Churned']))

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Stayed', 'Churned'],
    cmap='Blues', ax=ax
)
ax.set_title('Confusion Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color='#e74c3c', lw=2, label=f'ROC Curve (AUC = {auc_score:.4f})')
ax.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1, label='Random Baseline')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Random Forest', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Feature Importance

In [ ]:
# Feature importance as a DataFrame — actual numbers, not eyeballed from chart
feature_importance_df = pd.DataFrame({
    'Feature'    : X.columns,
    'Importance' : rf_model.feature_importances_
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print('Top 10 Features by Importance:')
print(feature_importance_df.head(10).to_string(index=False))

In [ ]:
# Feature importance plot
top15 = feature_importance_df.head(15)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=top15, x='Importance', y='Feature', palette='Reds_r', ax=ax)
ax.set_title('Top 15 Features by Importance', fontsize=14, fontweight='bold')
ax.set_xlabel('Relative Importance')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Threshold Tuning

Default threshold is **0.5**. Lower it to catch more churners (higher recall, lower precision). Raise it for fewer false alarms (higher precision, lower recall). Use the Precision/Recall chart below to decide.

In [ ]:
# THRESHOLD — adjust this value between 0.0 and 1.0
THRESHOLD = 0.5

y_pred_tuned = (y_prob >= THRESHOLD).astype(int)

print(f'Results at threshold = {THRESHOLD}:')
print(classification_report(y_test, y_pred_tuned, target_names=['Stayed', 'Churned']))

In [ ]:
# Precision / Recall tradeoff across all thresholds
precision, recall, thresh = precision_recall_curve(y_test, y_prob)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(thresh, precision[:-1], label='Precision', color='#3498db', lw=2)
ax.plot(thresh, recall[:-1],    label='Recall',    color='#e74c3c', lw=2)
ax.axvline(THRESHOLD, color='gray', linestyle='--', lw=1.5,
           label=f'Current threshold ({THRESHOLD})')
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title('Precision vs Recall at Different Thresholds', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('precision_recall_curve.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Save Trained Model

In [ ]:
joblib.dump(rf_model,       'churn_rf_model.pkl')
joblib.dump(label_encoders, 'label_encoders.pkl')

print('Model saved    → churn_rf_model.pkl')
print('Encoders saved → label_encoders.pkl')

---
## 11. Predict on Joined Customers

These are the 411 newly joined customers — the model scores each one and flags who is likely to churn.

In [ ]:
print(f'New customers to score: {join_data.shape[0]}')
join_data.head()

In [ ]:
# Keep original for output
original_join = join_data.copy()

# Prepare for prediction — drop non-feature columns
predict_input = join_data.drop(
    columns=['Customer_ID', 'Customer_Status', 'Churn_Category', 'Churn_Reason']
)

# Encode using the same encoders fitted on training data
for col in predict_input.select_dtypes(include=['object']).columns:
    if col in label_encoders:
        predict_input[col] = label_encoders[col].transform(predict_input[col])
    else:
        print(f'Warning: no encoder found for column "{col}" — skipping.')

# Score using tuned threshold
new_prob        = rf_model.predict_proba(predict_input)[:, 1]
new_predictions = (new_prob >= THRESHOLD).astype(int)

# Add results
original_join['Churn_Probability']         = new_prob.round(4)
original_join['Customer_Status_Predicted'] = new_predictions

# Filter to predicted churners, sorted by risk
churners = original_join[original_join['Customer_Status_Predicted'] == 1].copy()
churners = churners.sort_values('Churn_Probability', ascending=False)

print(f'Total customers scored : {len(original_join)}')
print(f'Predicted to churn     : {len(churners)}')
churners.head()

In [ ]:
# Save predictions — consumed by Power BI
churners.to_csv('Predictions.csv', index=False)

print('Predictions saved → Predictions.csv')
print(f'Rows exported     : {len(churners)}')